# Rotax 915 iS - The dataset

Is the simulated data complete, balanced and physically sensible, before any
model learns from it? One flight has three engines flown side by side: the faulty
one, a healthy twin (for residuals) and a wear-only one (for the health label).

Command-line equivalent: `validation/venv/bin/python validation_v5/run_v5.py 915`

In [ ]:
# Finds validation_v5 wherever Jupyter was started, then loads the helpers.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers_v5.py')): break
    if os.path.exists(os.path.join(d, 'validation_v5', 'nb_helpers_v5.py')):
        d = os.path.join(d, 'validation_v5'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers_v5 import *

ENGINE = '915'
setup(ENGINE)

### 1. What is in it

In [ ]:
data_overview(ENGINE)

### 2. The 14 checks
No NaN, instruments in range, split sizes, every sensor-fault cell filled, labels
consistent, healthy rows inside limits... Training refuses to start if one fails.

In [ ]:
ensure(ENGINE, 'checks')
print(load_json(ENGINE, 'data_checks.json'))

### 3. One flight, as the model sees it
Residuals sit on zero while the engine is healthy and move once a fault grows.
The bottom panel is the truth the model is trained to recover.

In [ ]:
g = example_flight(ENGINE)
plot_example_flight(g)

### 4. Build the training cache
Scales every input on training flights only, and stores 128-second windows every
16 s with their long-horizon context and labels. Run once (a few minutes).

In [ ]:
ensure(ENGINE, 'cache')

### What this means
If every check passes, the data is fit to train on: every fault and sensor-fault
kind has enough flights in each split to be learned and to be scored honestly.

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - what the model looks at for one answer. |
| **Healthy twin** | A perfectly healthy simulated copy of the engine flown through the same flight. |
| **Residual** | Reading minus the healthy twin's reading, in units of the sensor's noise (sigma). |
| **Train / cal / val / test** | Flights used to learn / to calibrate / to choose / to score. A flight is in one only. |
| **Baseline** | A do-nothing rule the model must beat by a clear margin, or its gate fails. |
| **95% interval** | The range the score would take on other test flights like these (flight bootstrap). |
| **TBO** | Time between overhauls: 2000 h (1200 h for the 915). |